# Who's Riley's Crush? — Google Colab runner

Serves the single static page in this repo and exposes it publicly with [localtunnel](https://github.com/localtunnel/localtunnel).

**Run the cells top to bottom.** The last cell prints a public `https://*.loca.lt` URL — that's the live page. It's safe to re-run that last cell any time (e.g. after pulling an update) — it automatically stops the previous run first.

## 1. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/DeQuackDealer/confess-web.git"
BRANCH = "claude/build-rileyscrush-f0276h"  # switch to "main" once this branch is merged
REPO_DIR = "/content/confess-web"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print("Repo already present, pulling latest...")
    !cd {REPO_DIR} && git pull

## 2. Install Node.js and localtunnel

The page itself is static (no Node needed to serve it — that's done with Python's built-in `http.server` below), but `localtunnel` is an npm package, so Node is needed for that.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!npm install -g localtunnel
!node -v && npm -v

## 3. Serve the page and open a public tunnel

Starts a static file server over the repo (so `index.html` is served at `/`) and localtunnel in the background, waits for both to report ready, then prints the public URL.

In [ ]:
import subprocess, os, time, threading, queue, signal, sys

PORT = 8000
REPO_DIR = "/content/confess-web"


def stop_previous_run():
    """Kills any server/tunnel left running from an earlier run of this cell in
    this same session, so re-running the cell doesn't fail with an
    address-already-in-use error. Each process is launched in its own session
    (start_new_session) and stopped via the whole process group, not just one
    PID, since some of these commands spawn further child processes."""
    for name in ("tunnel_proc", "server_proc"):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except ProcessLookupError:
                    pass
                proc.wait(timeout=5)
            print(f"Stopped previous {name}.")


def start_process(cmd, cwd=None, env=None):
    return subprocess.Popen(
        cmd, cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        start_new_session=True,
    )


def pump_output(proc, label, q):
    """Continuously drains a process's stdout so it never blocks on a full pipe,
    prints each line with a label, and forwards it to a queue for the marker wait."""
    for line in proc.stdout:
        print(f"[{label}] {line}", end="")
        q.put(line)
    q.put(None)


def wait_for_marker(q, marker, timeout=60):
    deadline = time.time() + timeout
    while True:
        remaining = deadline - time.time()
        if remaining <= 0:
            return False
        try:
            line = q.get(timeout=remaining)
        except queue.Empty:
            return False
        if line is None:
            return False
        if marker in line:
            return True


stop_previous_run()

print("Starting static file server...")
server_q = queue.Queue()
server_proc = start_process(
    [sys.executable, "-m", "http.server", str(PORT), "--directory", REPO_DIR],
)
threading.Thread(target=pump_output, args=(server_proc, "server", server_q), daemon=True).start()

if not wait_for_marker(server_q, "Serving HTTP", timeout=30):
    print("\n\u26a0\ufe0f Server did not report ready in time \u2014 check the log above for errors.")
else:
    print("\nStarting localtunnel...")
    tunnel_q = queue.Queue()
    tunnel_proc = start_process(["lt", "--port", str(PORT)])
    threading.Thread(target=pump_output, args=(tunnel_proc, "localtunnel", tunnel_q), daemon=True).start()

    if wait_for_marker(tunnel_q, "your url is", timeout=60):
        print("\n\u2705 The page is live at the URL printed above.")
        print("   First-time visitors may see localtunnel's interstitial page \u2014 click 'Click to Continue'.")
    else:
        print("\n\u26a0\ufe0f localtunnel did not report a URL in time \u2014 check the log above for errors.")

## 4. (Optional) Stop everything

Run this cell to shut down the server and tunnel without immediately starting a new run (cell 3 already does this automatically at the start of every run).

In [ ]:
stop_previous_run()